# 🏦 Credit Card Application Review with Azure AI Agents Streaming

## Overview

This notebook demonstrates **Azure AI Agents** with **streaming** for a credit card application review workflow. Watch real-time analysis as two specialized agents process an application.

### 💼 Industry Use Case: Credit Card Application Pipeline

A customer submits a credit card application:
1. **Credit Analyst Agent** - Reviews application and assesses creditworthiness
2. **Underwriter Agent** - Makes final decision with terms and conditions (streaming output)

### ⚠️ Important Financial Disclaimer
> **This notebook is for educational purposes only.** The credit decision logic is simplified and should not be used for actual lending decisions. Always follow regulatory requirements and consult compliance teams.

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Azure AI Agent Service** | Using Azure-hosted agents with the Agent Framework |
| **Streaming** | Real-time token generation via `AgentRunUpdateEvent` |
| **register_agent()** | Lazy agent creation with factory functions |
| **WorkflowBuilder** | Pipeline construction with edges between agents |

### Architecture

```
Credit Card Application
    ↓
Credit Analyst Agent (creditworthiness assessment)
    ↓ streaming
Underwriter Agent (approval decision + terms)
    ↓ streaming
Final Credit Decision
```

## Prerequisites

- ✅ Azure AI Foundry configured
- ✅ Environment variables: `AI_FOUNDRY_PROJECT_ENDPOINT`, `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- ✅ Azure CLI authentication: Run `az login`

## 1️⃣ Import Libraries and Load Environment

In [ ]:
# Copyright (c) Microsoft. All rights reserved.
import os
import sys
from importlib.metadata import version
from pathlib import Path

from agent_framework import Agent, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

assert version("agent-framework-core") == "1.17.0", "Select the pinned project kernel."
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements.in").is_file())
assert Path(sys.executable).resolve() == (repo_root / ".venv/Scripts/python.exe").resolve(), (
    "Select the repository .venv kernel."
)
load_dotenv(repo_root / ".env", override=False)

PROJECT_ENDPOINT = (
    os.getenv("FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AZURE_AI_PROJECT_ENDPOINT")
)
MODEL_DEPLOYMENT = os.getenv("FOUNDRY_MODEL") or os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
print("✅ Libraries imported and environment loaded")


## 2️⃣ Define Agent Factory Functions

The factory function pattern ensures proper resource management:

1. **Lazy Initialization**: Agents created only when client is ready
2. **Shared Session**: All agents use the same authenticated client
3. **Clean Separation**: Agent config separate from workflow logic

### 📊 Credit Analyst Agent
- Reviews application financials
- Assesses debt-to-income ratio
- Evaluates credit history factors

### ✅ Underwriter Agent  
- Makes approval/denial decision
- Sets credit limit and APR
- Provides terms and conditions

In [ ]:
def create_credit_analyst(client: FoundryChatClient) -> Agent:
    """Creates a Credit Analyst agent for application review."""
    return Agent(
        client=client,
        name="CreditAnalyst",
        instructions=(
            "You are a Credit Analyst at a retail bank reviewing credit card applications.\n"
            "For each application:\n"
            "1. Evaluate the applicant's income and employment stability\n"
            "2. Assess debt-to-income ratio\n"
            "3. Review credit score and history factors\n"
            "4. Identify any risk factors or concerns\n"
            "5. Provide a preliminary creditworthiness assessment\n"
            "Be thorough but concise in your analysis."
        ),
    )


def create_underwriter(client: FoundryChatClient) -> Agent:
    """Creates an Underwriter agent for final credit decisions."""
    return Agent(
        client=client,
        name="Underwriter",
        instructions=(
            "You are a Credit Card Underwriter at a retail bank.\n"
            "Based on the Credit Analyst's assessment, make a final decision:\n"
            "1. APPROVE, DECLINE, or CONDITIONAL APPROVAL\n"
            "2. If approved, recommend credit limit ($500-$15,000 range)\n"
            "3. If approved, recommend APR range based on risk\n"
            "4. List any conditions for conditional approvals\n"
            "5. Include standard regulatory disclosures\n"
            "Provide a clear, structured decision."
        ),
    )

## 3️⃣ Build and Run the Credit Application Workflow

### Workflow Configuration (Agent Framework 1.17.0)
- **`WorkflowBuilder(start_executor=...)`**: the Credit Analyst is the entry point (constructor argument)
- **`output_from="all"`**: stream every agent's tokens as workflow output events
- **`add_edge(analyst, underwriter)`**: flow from analyst → underwriter
- **Agents as executors**: `Agent` instances can be used directly as workflow nodes

### Streaming Events
- **`workflow.run(..., stream=True)`**: yields `WorkflowEvent` objects
- **`event.type == "output"`** with **`AgentResponseUpdate`** data: real-time agent tokens
- **`event.executor_id`**: identifies which agent produced each chunk


In [ ]:
async def main() -> None:
    with AzureCliCredential() as credential:
        client = FoundryChatClient(
            project_endpoint=PROJECT_ENDPOINT,
            model=MODEL_DEPLOYMENT,
            credential=credential,
        )
        try:
            credit_analyst = create_credit_analyst(client)
            underwriter = create_underwriter(client)

            # Build the credit card application workflow.
            # output_from="all" streams every agent's tokens as workflow output events.
            workflow = (
                WorkflowBuilder(start_executor=credit_analyst, output_from="all")
                .add_edge(credit_analyst, underwriter)
                .build()
            )

            # Sample credit card application
            credit_application = """
            CREDIT CARD APPLICATION
            =======================
            Applicant: Sarah Johnson
            Application Date: 2024-01-15
            Product: Rewards Visa Card

            PERSONAL INFORMATION:
            - Age: 32
            - Residence: Homeowner (5 years)

            FINANCIAL INFORMATION:
            - Annual Income: $85,000
            - Employment: Marketing Manager at Tech Corp (4 years)
            - Monthly Rent/Mortgage: $1,800
            - Existing Credit Cards: 2 (total limit $12,000, utilization 25%)
            - Auto Loan Balance: $15,000 (monthly payment $350)
            - Credit Score: 745

            REQUESTED CREDIT LIMIT: $10,000
            """

            print("💳 CREDIT CARD APPLICATION REVIEW")
            print("=" * 60)
            print(credit_application)
            print("=" * 60 + "\n")

            last_executor_id: str | None = None

            # In 1.17.0, workflow.run(..., stream=True) yields WorkflowEvent objects.
            # Agent tokens arrive as event.type == "output" with AgentResponseUpdate data.
            events = workflow.run(credit_application, stream=True)
            async for event in events:
                if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
                    eid = event.executor_id or ""
                    if eid != last_executor_id:
                        if last_executor_id is not None:
                            print()
                        agent_emoji = "📊" if "analyst" in eid.lower() else "✅"
                        print(f"\n{agent_emoji} {eid.upper()}:", end=" ", flush=True)
                        last_executor_id = eid
                    print(event.data.text or "", end="", flush=True)

            print("\n" + "=" * 60)
            print("✅ Credit card application review complete!")
            print("\n⚠️ DISCLAIMER: This is a demonstration only. Actual credit")
            print("   decisions require full underwriting and regulatory compliance.")
        finally:
            await client.client.close()
            await client.project_client.close()


await main()


## 📝 Key Takeaways

### Azure AI Agents for Credit Decisioning

| Concept | Description |
|---------|-------------|
| **Agents as executors** | Pass `Agent` instances directly to `WorkflowBuilder` |
| **`start_executor` / `output_from`** | Constructor arguments that define entry point and outputs |
| **Streaming Events** | `event.type == "output"` with `AgentResponseUpdate` for real-time tokens |
| **Resource cleanup** | `try/finally` closes the `FoundryChatClient` |

### FSI Benefits of Streaming Workflows

| Benefit | Application |
|---------|-------------|
| **Transparency** | Show reasoning process to compliance teams |
| **Audit Trail** | Record decision-making flow |
| **Early Detection** | Catch issues before final decision |
| **Customer Experience** | Faster perceived response times |

### Next Steps
- Add **Credit Bureau Tool** for real credit data
- Implement **Human-in-the-Loop** for edge cases
- Add **Compliance Executor** for required disclosures
